# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.1 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID='task089'
TASK_PATH=Path(COMPETITION)/'task089.json'
OUT_DIR=Path.cwd()/'task089_zero_pad'
os.makedirs(OUT_DIR, exist_ok=True)
ONNX_PATH=os.path.join(OUT_DIR, f'{TASK_ID}.onnx')
ZIP_PATH=Path.cwd()/'submission.zip'
STATIC_ZIP=os.path.join(OUT_DIR, f'{TASK_ID}_zero_pad_static_graph_submission.zip')
NB_PATH=OUT_DIR/f'{TASK_ID}_zero_pad_solver.ipynb'
NB_EXEC=OUT_DIR/f'{TASK_ID}_zero_pad_solver_executed.ipynb'
AUDIT_JSON=OUT_DIR/f'{TASK_ID}_zero_pad_audit.json'
AUDIT_CSV=OUT_DIR/f'{TASK_ID}_zero_pad_audit.csv'

In [6]:
def shift_lookup(a, dy:int, dx:int):
    # result[r,c] = a[r+dy,c+dx], zeros outside. Static, no Pad/Shape.
    z=a*0.0
    # vertical
    if dy>0:
        b=torch.cat([a[:,:,dy:,:], z[:,:,:dy,:]], dim=2)
    elif dy<0:
        k=-dy
        b=torch.cat([z[:,:,:k,:], a[:,:,:30-k,:]], dim=2)
    else:
        b=a
    # horizontal
    if dx>0:
        c=torch.cat([b[:,:,:,dx:], z[:,:,:,:dx]], dim=3)
    elif dx<0:
        k=-dx
        c=torch.cat([z[:,:,:,:k], b[:,:,:,:30-k]], dim=3)
    else:
        c=b
    return c

class Task089TemplateCopy(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        # x [1,10,30,30], real canvas one-hot, padded area zero-vector
        active=torch.clamp(x.sum(dim=1, keepdim=True),0,1)
        out_ch=[]
        # Source marker if adjacent to that color. Copy offset template to isolated markers.
        for c in range(1,10):
            col=x[:,c:c+1,:,:] * active
            gen_total=col*0.0
            for m in [2,3]:
                if c==m:
                    continue
                mark=x[:,m:m+1,:,:] * active
                adj=col*0.0
                for ay in [-1,0,1]:
                    for ax in [-1,0,1]:
                        if ay==0 and ax==0: continue
                        adj=adj+shift_lookup(col, ay, ax)
                adj=(adj>0.5).float()
                source=mark*adj
                target=mark*(1.0-adj)
                gen=col*0.0
                for dy in [-2,-1,0,1,2]:
                    for dx in [-2,-1,0,1,2]:
                        present=torch.clamp((source*shift_lookup(col, dy, dx)).sum(dim=(2,3), keepdim=True),0,1)
                        tdy=dy
                        tdx=-dx if m==2 else dx
                        gen=gen+present*shift_lookup(target, -tdy, -tdx)
                gen_total=gen_total+gen
            out_ch.append(torch.clamp(col+gen_total,0,1))
        fg=torch.cat(out_ch, dim=1) * active
        occupied=torch.clamp(fg.sum(dim=1, keepdim=True),0,1)
        bg=active*(1.0-occupied)
        return torch.cat([bg,fg], dim=1)

def grid_to_tensor(grid):
    arr=np.array(grid,dtype=np.int64)
    h,w=arr.shape
    x=np.zeros((1,10,30,30), dtype=np.float32)
    for c in range(10):
        x[0,c,:h,:w]=(arr==c)
    return x

def target_tensor(grid):
    return grid_to_tensor(grid)

def tensor_to_grid(y, shape):
    pred=y[0].argmax(axis=0).astype(np.int64)
    return pred[:shape[0],:shape[1]]

def raw_match(y, grid):
    return np.array_equal((y>0.5).astype(np.float32), target_tensor(grid))

def verify_model_ort(sess, data):
    rows=[]
    for split in ['train','test','arc-gen']:
        ok=0; n=0; raw_ok=0
        for i,ex in enumerate(data[split]):
            x=grid_to_tensor(ex['input'])
            y=sess.run(None, {'input':x})[0]
            out=np.array(ex['output'])
            pred=tensor_to_grid(y,out.shape)
            aok = pred.shape==out.shape and np.array_equal(pred,out)
            rok = raw_match(y, ex['output'])
            ok += int(aok); raw_ok += int(rok); n+=1
        rows.append({'split':split,'argmax_ok':ok,'raw_ok':raw_ok,'total':n})
    return rows

def forbidden_ops(model):
    forbidden={'Loop','Scan','NonZero','Unique','Script','Function','TreeEnsembleClassifier','TreeEnsembleRegressor'}
    risky={'Shape','Gather','ConstantOfShape','Expand','Range','ScatterND'}
    ops=Counter(n.op_type for n in model.graph.node)
    return ops, sorted(set(ops)&forbidden), sorted(set(ops)&risky)

def build():
    data=json.load(open(TASK_PATH))
    model=Task089TemplateCopy().eval()
    dummy=np.zeros((1,10,30,30), np.float32)
    # Use a visible input as dummy so type/signature stable, but shape fixed
    dummy=grid_to_tensor(data['train'][0]['input'])
    torch.onnx.export(model, torch.from_numpy(dummy), ONNX_PATH, input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True, dynamo=False)
    onnx_model=onnx.load(ONNX_PATH)
    onnx.checker.check_model(onnx_model)
    ops,bad,risky=forbidden_ops(onnx_model)
    sess=ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    rows=verify_model_ort(sess,data)
    hold=data['arc-gen'][int(len(data['arc-gen'])*0.4):]
    hold_ok=hold_raw=0
    for ex in hold:
        y=sess.run(None, {'input':grid_to_tensor(ex['input'])})[0]
        hold_ok += int(np.array_equal(tensor_to_grid(y, np.array(ex['output']).shape), np.array(ex['output'])))
        hold_raw += int(raw_match(y, ex['output']))
    # OOD generator for same rule
    import random
    random.seed(0)
    def apply_rule_np(inp):
        g=np.array(inp); h,w=g.shape; out=g.copy()
        def sh(arr,dy,dx):
            res=np.zeros_like(arr,bool); r0=max(0,-dy); r1=min(h,h-dy); c0=max(0,-dx); c1=min(w,w-dx)
            if r1>r0 and c1>c0: res[r0:r1,c0:c1]=arr[r0+dy:r1+dy,c0+dx:c1+dx]
            return res
        for m in [2,3]:
            mark=(g==m)
            for c in range(1,10):
                if c==m: continue
                col=(g==c)
                adj=np.zeros_like(mark,bool)
                for ay in [-1,0,1]:
                    for ax in [-1,0,1]:
                        if ay or ax: adj |= sh(col,ay,ax)
                source=mark & adj; target=mark & (~adj)
                if not source.any() or not target.any(): continue
                for dy in [-2,-1,0,1,2]:
                    for dx in [-2,-1,0,1,2]:
                        if (source & sh(col,dy,dx)).any():
                            tdx=-dx if m==2 else dx
                            for r,cc in np.argwhere(target):
                                rr=r+dy; jj=cc+tdx
                                if 0<=rr<h and 0<=jj<w and out[rr,jj]==0: out[rr,jj]=c
        return out.tolist()
    ood=0;oodok=0;oodraw=0
    motifs=[[(0,1),(1,0),(1,1),(2,2)],[(-1,-1),(-1,0),(-1,1),(0,1),(1,1)],[(-2,-1),(-2,0),(-1,0),(0,-1),(0,1)],[(0,-1),(1,0),(1,1)]]
    for h in [9,13,17,25,30]:
      for w in [9,13,19,30]:
        for m in [2,3]:
          for c in [1,4,5,8,9]:
            if c==m: continue
            for motif in motifs[:3]:
              # choose source and target positions with margin 2
              g=np.zeros((h,w),int)
              sy=random.randint(2,h-3); sx=random.randint(2,w-3)
              g[sy,sx]=m
              ok=True
              for dy,dx in motif:
                yy=sy+dy; xx=sx+dx
                if not(0<=yy<h and 0<=xx<w): ok=False
                else: g[yy,xx]=c
              if not ok: continue
              for _ in range(2):
                ty=random.randint(2,h-3); tx=random.randint(2,w-3)
                if abs(ty-sy)+abs(tx-sx)>4: g[ty,tx]=m
              yref=np.array(apply_rule_np(g.tolist()))
              y=sess.run(None, {'input':grid_to_tensor(g.tolist())})[0]
              ood+=1
              oodok+=int(np.array_equal(tensor_to_grid(y,yref.shape),yref))
              oodraw+=int(raw_match(y,yref.tolist()))
    audit={'task':TASK_ID,'onnx_size':os.path.getsize(ONNX_PATH),'ops':dict(ops),'forbidden_ops':bad,'risky_ops':risky,'rows':rows,'holdout_argmax_ok':hold_ok,'holdout_raw_ok':hold_raw,'holdout_total':len(hold),'ood_ok':oodok,'ood_raw_ok':oodraw,'ood_total':ood}
    json.dump(audit, open(AUDIT_JSON,'w'), indent=2)
    with open(AUDIT_CSV,'w',newline='') as f:
        wr=csv.writer(f); wr.writerow(['metric','value'])
        wr.writerow(['task',TASK_ID]); wr.writerow(['onnx_size',audit['onnx_size']]); wr.writerow(['forbidden_ops',','.join(bad)]); wr.writerow(['risky_ops',','.join(risky)])
        for r in rows:
            wr.writerow([r['split']+'_argmax',f"{r['argmax_ok']}/{r['total']}"]); wr.writerow([r['split']+'_raw',f"{r['raw_ok']}/{r['total']}"])
        wr.writerow(['holdout_argmax',f'{hold_ok}/{len(hold)}']); wr.writerow(['holdout_raw',f'{hold_raw}/{len(hold)}']); wr.writerow(['ood_argmax',f'{oodok}/{ood}']); wr.writerow(['ood_raw',f'{oodraw}/{ood}'])
    
    for zp in [ZIP_PATH,STATIC_ZIP]:
        with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z: z.write(ONNX_PATH, f'{TASK_ID}.onnx')
    
    return audit




In [7]:
audit=build()
audit


/tmp/ipykernel_16/249131228.py:104: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.from_numpy(dummy), ONNX_PATH, input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True, dynamo=False)


{'task': 'task089',
 'onnx_size': 662915,
 'ops': {'Constant': 3704,
  'ReduceSum': 402,
  'Clip': 411,
  'Slice': 709,
  'Mul': 868,
  'Concat': 602,
  'Add': 497,
  'Greater': 9,
  'Cast': 9,
  'Sub': 10},
 'forbidden_ops': [],
 'risky_ops': [],
 'rows': [{'split': 'train', 'argmax_ok': 4, 'raw_ok': 4, 'total': 4},
  {'split': 'test', 'argmax_ok': 1, 'raw_ok': 1, 'total': 1},
  {'split': 'arc-gen', 'argmax_ok': 262, 'raw_ok': 262, 'total': 262}],
 'holdout_argmax_ok': 158,
 'holdout_raw_ok': 158,
 'holdout_total': 158,
 'ood_ok': 597,
 'ood_raw_ok': 582,
 'ood_total': 600}